# Nemotron Clinical Insight Engine
### NVIDIA Nemotron-3-Nano · Patient Deterioration Early Warning
---
**What this notebook does:**
1.  **Connection test** — verifies your local llama.cpp Nemotron server is running
2.  **Smoke test** — confirms the model responds before touching patient data
3.  **Single patient** — deep-dive clinical summary with full SHAP context
4.  **Batch summaries** — generates insights for all high/moderate risk patients
5.  **Prompt engineering** — compares 4 prompt styles to find the best output
6.  **Base vs. Instruct** — side-by-side output comparison
7.  **Export** — saves summaries to CSV, ready for the dashboard

---
> **Prerequisite:** Run `01_patient_deterioration.ipynb` first.
>
> **Server command (Base):**
> ```
> llama-server --model ~/nemotron-nano/*.gguf --port 8001 --ctx-size 8192
> ```

In [1]:
import pandas as pd
import numpy as np
import json, os, sys, time, textwrap, warnings
from pathlib import Path
from openai import OpenAI
import requests

warnings.filterwarnings('ignore')

# ─── Configuration ────────────────────────────────────────────────────────
# Switch between 'base' and 'instruct' here

MODEL_TYPE    = 'base'     # 'base' or 'instruct'
BASE_PORT     = 8001       # llama-server port for Base model
INSTRUCT_PORT = 8002       # llama-server port for Instruct model

ACTIVE_PORT = BASE_PORT if MODEL_TYPE == 'base' else INSTRUCT_PORT
BASE_URL    = f'http://localhost:{ACTIVE_PORT}/v1'

DATA_DIR  = Path('../data')
MODEL_DIR = Path('../model')

print('=' * 58)
print('  Nemotron Clinical Insight Engine')
print('=' * 58)
print(f'  Model type  : {MODEL_TYPE.upper()}')
print(f'  Server URL  : {BASE_URL}')
print(f'  Data dir    : {DATA_DIR.resolve()}')
print()
print('  Server start command:')
if MODEL_TYPE == 'base':
    print(f'    llama-server --model ~/nemotron-nano/*.gguf --port {BASE_PORT} --ctx-size 8192')
else:
    print(f'    llama-server --model ~/nemotron-nano-instruct/*.gguf --port {INSTRUCT_PORT} --ctx-size 8192 --chat-template chatml')
print('=' * 58)

  Nemotron Clinical Insight Engine
  Model type  : BASE
  Server URL  : http://localhost:8001/v1
  Data dir    : /Users/smonroe/nvidia-health-demo/data

  Server start command:
    llama-server --model ~/nemotron-nano/*.gguf --port 8001 --ctx-size 8192


In [2]:
def check_server(port, label=''):
    try:
        r = requests.get(f'http://localhost:{port}/health', timeout=3)
        if r.status_code == 200:
            status = r.json().get('status', 'ok')
            print(f'   {label} (port {port}): {status.upper()}')
            return True
        print(f'   {label} (port {port}): HTTP {r.status_code}')
        return False
    except requests.exceptions.ConnectionError:
        print(f'   {label} (port {port}): NOT RUNNING')
        print(f'     Start: llama-server --model ~/nemotron-nano/*.gguf --port {port}')
        return False
    except Exception as e:
        print(f'   {label} (port {port}): {e}')
        return False

print('Checking Nemotron server connections...')
print()
base_ok     = check_server(BASE_PORT,     'Base model    ')
instruct_ok = check_server(INSTRUCT_PORT, 'Instruct model')
print()
if MODEL_TYPE == 'base' and not base_ok:
    print('⚠️  Base server not running. Start it before continuing.')
elif MODEL_TYPE == 'instruct' and not instruct_ok:
    print('⚠️  Instruct server not running. Start it before continuing.')
else:
    print(f' Active server ({MODEL_TYPE.upper()}) is ready.')

Checking Nemotron server connections...

   Base model     (port 8001): HTTP 401
   Instruct model (port 8002): NOT RUNNING
     Start: llama-server --model ~/nemotron-nano/*.gguf --port 8002

⚠️  Base server not running. Start it before continuing.


In [3]:
client = OpenAI(base_url=BASE_URL, api_key='not-required')

def call_base(prompt, max_tokens=120, temperature=0.2, stop=None):
    stop = stop or ['\n\n', 'Patient:', '===']
    resp = client.completions.create(
        model='local-model', prompt=prompt,
        max_tokens=max_tokens, temperature=temperature, stop=stop
    )
    return resp.choices[0].text.strip()

def call_instruct(system, user, max_tokens=200, temperature=0.3):
    resp = client.chat.completions.create(
        model='local-model',
        messages=[{'role':'system','content':system},{'role':'user','content':user}],
        max_tokens=max_tokens, temperature=temperature
    )
    return resp.choices[0].message.content.strip()

def call_model(prompt_or_user, system=None, **kwargs):
    if MODEL_TYPE == 'instruct':
        sys_msg = system or 'You are a helpful clinical AI. Be concise and specific.'
        return call_instruct(sys_msg, prompt_or_user, **kwargs)
    return call_base(prompt_or_user, **kwargs)

# Test 1: basic clinical knowledge
print('Test 1: Basic clinical knowledge...')
t0 = time.time()
if MODEL_TYPE == 'instruct':
    r1 = call_instruct(
        system='You are a helpful assistant. Answer in one sentence.',
        user='What does SpO2 below 94% indicate in an ICU patient?',
        max_tokens=80
    )
else:
    raw = call_base('In ICU medicine, a SpO2 below 94% indicates', max_tokens=60, stop=['\n', '.'])
    r1  = 'SpO2 below 94% indicates ' + raw
print(f'   {time.time()-t0:.1f}s  |  {r1[:100]}')
print()

# Test 2: structured clinical output
print('Test 2: Structured clinical summary...')
t0 = time.time()
if MODEL_TYPE == 'instruct':
    r2 = call_instruct(
        system='You are a clinical AI. Respond in exactly 2 sentences.',
        user='Summarize risk for: HR 118, SpO2 91%, SBP 88, Lactate 4.2.',
        max_tokens=100
    )
else:
    raw2 = call_base(
        'ICU Risk Summary\nVitals: HR 118 | SpO2 91% | SBP 88 | Lactate 4.2\nAssessment: This patient shows signs of',
        max_tokens=80, stop=['\n\n', 'Vitals:']
    )
    r2 = 'This patient shows signs of ' + raw2
print(f'   {time.time()-t0:.1f}s  |  {r2[:120]}')
print()
print(' Smoke tests passed — Nemotron is responding correctly.')

Test 1: Basic clinical knowledge...


AuthenticationError: Error code: 401 - {'error': {'code': 'INVALID_API_KEY', 'message': 'Invalid API key provided'}}

In [ ]:
print('Loading outputs from notebook 01...')

try:
    df          = pd.read_csv(DATA_DIR / 'patients_with_risk.csv')
    alert_queue = pd.read_csv(DATA_DIR / 'alert_queue.csv')
    with open(DATA_DIR / 'shap_metadata.json') as f:
        shap_meta = json.load(f)
    with open(DATA_DIR / 'feature_cols.json') as f:
        feat_meta = json.load(f)
    FEATURE_DISPLAY = feat_meta['feature_display']
    FEATURE_COLS    = feat_meta['feature_cols']
    print(f'   Patients      : {len(df):,}')
    print(f'   Alert queue   : {len(alert_queue)} high-risk patients')
    print(f'   Top drivers   : {", ".join(shap_meta["top5_features"][:3])}')
except FileNotFoundError as e:
    print(f'   Missing: {e}')
    print('     Run 01_patient_deterioration.ipynb first.')
    raise

def get_top_drivers(patient_id, n=3):
    row     = df[df['patient_id'] == patient_id].iloc[0]
    drivers = []
    for feat_name, feat_col in zip(FEATURE_DISPLAY, FEATURE_COLS):
        if feat_col in df.columns:
            val = float(row[feat_col])
            mx  = float(df[feat_col].max())
            mn  = float(df[feat_col].min())
            rng = mx - mn
            if rng > 0:
                drivers.append((feat_name, (val - mn) / rng))
    drivers.sort(key=lambda x: x[1], reverse=True)
    return [name for name, _ in drivers[:n]]

print()
print('Risk distribution:')
print(df['risk_level'].value_counts().to_string())
print()
print(' Data loaded successfully.')

In [ ]:
def build_patient_context(row):
    drivers  = get_top_drivers(row['patient_id'], n=3)
    abnormal = []
    if float(row['hr'])   > 100: abnormal.append(f'tachycardia HR {row["hr"]:.0f}')
    if float(row['spo2']) < 94:  abnormal.append(f'hypoxia SpO2 {row["spo2"]:.1f}%')
    if float(row['sbp'])  < 100: abnormal.append(f'hypotension SBP {row["sbp"]:.0f}')
    if float(row['rr'])   > 20:  abnormal.append(f'tachypnea RR {row["rr"]:.0f}')
    if float(row.get('lactate', 0)) > 2: abnormal.append(f'lactate {row["lactate"]:.1f}')
    return {
        'patient_id': row['patient_id'], 'name': row['name'],
        'age': int(row['age']),          'gender': row['gender'],
        'diagnosis': row['diagnosis'],   'room': row.get('room', '-'),
        'risk_score': float(row['risk_score']),
        'risk_level': str(row['risk_level']),
        'drivers': drivers,
        'abnormal': ', '.join(abnormal) if abnormal else 'none',
        'hr': float(row['hr']),           'spo2': float(row['spo2']),
        'sbp': float(row['sbp']),         'rr': float(row['rr']),
        'temp': float(row.get('temp', 37.0)),
        'lactate': float(row.get('lactate', 0)),
        'wbc': float(row.get('wbc', 0)),
        'creatinine': float(row.get('creatinine', 0)),
        'news2': int(row.get('news2_score', 0)),
        'flags': int(row.get('combined_flag_count', 0)),
        'trend': float(row.get('trend_severity', 0)),
        'hours_in': int(row.get('hours_admitted', 0)),
    }

def build_base_prompt(ctx):
    lines = [
        'ICU CLINICAL AI RISK SUMMARY',
        '=' * 50,
        f'Patient:     {ctx["name"]} | Age {ctx["age"]} {ctx["gender"]} | Room {ctx["room"]}',
        f'Diagnosis:   {ctx["diagnosis"]}',
        f'Admitted:    {ctx["hours_in"]} hours ago',
        '-' * 50,
        f'RISK SCORE:  {ctx["risk_score"]:.0%}  [{ctx["risk_level"]}]',
        f'NEWS2:       {ctx["news2"]}  |  Active flags: {ctx["flags"]}/7',
        f'Top Drivers: {", ".join(ctx["drivers"])}',
        '-' * 50,
        'VITALS:',
        f'  Heart Rate:  {ctx["hr"]:.0f} bpm',
        f'  SpO2:        {ctx["spo2"]:.1f}%',
        f'  Systolic BP: {ctx["sbp"]:.0f} mmHg',
        f'  Resp Rate:   {ctx["rr"]:.0f} /min',
        f'  Temperature: {ctx["temp"]:.1f} C',
        'LABS:',
        f'  Lactate:     {ctx["lactate"]:.1f} mmol/L',
        f'  WBC:         {ctx["wbc"]:.1f} k/uL',
        f'  Creatinine:  {ctx["creatinine"]:.2f} mg/dL',
        '-' * 50,
        f'ABNORMAL FLAGS: {ctx["abnormal"]}',
        '-' * 50,
        'CLINICAL SUMMARY:',
        '  Assessment:  This patient presents with',
    ]
    return '\n'.join(lines)

def build_instruct_prompt(ctx):
    d_str  = '\n'.join(f'  {i+1}. {d}' for i, d in enumerate(ctx['drivers']))
    system = (
        'You are a clinical AI assistant for an ICU care team. '
        'Generate concise, accurate, actionable patient risk summaries. '
        'Always include: (1) key clinical concern, (2) supporting evidence, '
        '(3) specific recommended action with timeframe. '
        'Plain English — no unnecessary jargon.'
    )
    user = (
        f'Generate a 3-sentence clinical summary for the ICU care team.\n\n'
        f'PATIENT: {ctx["name"]}, Age {ctx["age"]} {ctx["gender"]}\n'
        f'DIAGNOSIS: {ctx["diagnosis"]} | Admitted {ctx["hours_in"]}h ago\n'
        f'RISK: {ctx["risk_score"]:.0%} ({ctx["risk_level"]}) | NEWS2: {ctx["news2"]} | {ctx["flags"]}/7 flags\n\n'
        f'PRIMARY RISK DRIVERS:\n{d_str}\n\n'
        f'VITALS: HR {ctx["hr"]:.0f} | SpO2 {ctx["spo2"]:.1f}% | SBP {ctx["sbp"]:.0f} | RR {ctx["rr"]:.0f}\n'
        f'LABS: Lactate {ctx["lactate"]:.1f} | WBC {ctx["wbc"]:.1f} | Creatinine {ctx["creatinine"]:.2f}\n\n'
        f'ABNORMAL FLAGS: {ctx["abnormal"]}\n\nWrite the 3-sentence summary now:'
    )
    return system, user

def generate_insight(row, verbose=False):
    ctx = build_patient_context(row)
    t0  = time.time()
    if MODEL_TYPE == 'instruct':
        system, user = build_instruct_prompt(ctx)
        result = call_instruct(system, user, max_tokens=220, temperature=0.3)
    else:
        prompt = build_base_prompt(ctx)
        raw    = call_base(prompt, max_tokens=200, temperature=0.2,
                           stop=['\n\nICU', '\n\n=', 'Patient:'])
        result = 'This patient presents with ' + raw
    if verbose:
        print(f'  Generated in {time.time()-t0:.1f}s')
    return result

# Run on highest-risk patient
top = df.nlargest(1, 'risk_score').iloc[0]
print(f'Deep-dive: {top["patient_id"]} — {top["name"]} — Risk {top["risk_score"]:.0%}')
print()
insight = generate_insight(top, verbose=True)
print('─' * 65)
print(f'   {top["patient_id"]}  {top["name"]}  Age {int(top["age"])}  |  {top["diagnosis"]}')
print(f'  Risk: {top["risk_score"]:.0%}  |  NEWS2: {int(top["news2_score"])}')
print()
print(f'   Nemotron ({MODEL_TYPE.upper()}):')
print()
for line in textwrap.wrap(insight, 63):
    print(f'     {line}')
print('─' * 65)

In [ ]:
high_risk = df[df['risk_level'].isin(['HIGH','MODERATE'])].copy()
high_risk = high_risk.sort_values('risk_score', ascending=False)

print(f'Generating summaries for {len(high_risk)} patients...')
print(f'Model: {MODEL_TYPE.upper()} | {BASE_URL}')
print()

results = []
errors  = []
t_batch = time.time()

for i, (_, row) in enumerate(high_risk.iterrows()):
    try:
        t0      = time.time()
        insight = generate_insight(row)
        elapsed = time.time() - t0
        results.append({
            'patient_id':       row['patient_id'],
            'name':             row['name'],
            'age':              int(row['age']),
            'diagnosis':        row['diagnosis'],
            'risk_score':       float(row['risk_score']),
            'risk_level':       str(row['risk_level']),
            'news2_score':      int(row.get('news2_score', 0)),
            'hr':               float(row['hr']),
            'spo2':             float(row['spo2']),
            'sbp':              float(row['sbp']),
            'nemotron_insight': insight,
            'model_type':       MODEL_TYPE,
            'generation_time_s': round(elapsed, 2),
        })
        icon = '🔴' if row['risk_level'] == 'HIGH' else '🟡'
        print(f'  {icon} [{i+1:>3}/{len(high_risk)}] {row["patient_id"]}  '
              f'{str(row["name"]):<22}  {row["risk_score"]:.0%}  ({elapsed:.1f}s)')
    except Exception as e:
        errors.append({'patient_id': row['patient_id'], 'error': str(e)})
        print(f'   [{i+1:>3}] {row["patient_id"]} — {e}')

insights_df = pd.DataFrame(results)
total_time  = time.time() - t_batch
print()
print('=' * 55)
print(f'  Summaries  : {len(results)}')
print(f'  Errors     : {len(errors)}')
print(f'  Total time : {total_time:.1f}s')
if results:
    print(f'  Avg/patient: {insights_df["generation_time_s"].mean():.1f}s')
print('=' * 55)

In [ ]:
test_row = df[df['risk_level'] == 'HIGH'].nlargest(1, 'risk_score').iloc[0]
ctx      = build_patient_context(test_row)
print(f'Test patient: {ctx["patient_id"]} — {ctx["name"]} — {ctx["risk_score"]:.0%}')
print()

if MODEL_TYPE == 'instruct':
    variants = {
        'A: Minimal': (
            'You are a clinical assistant.',
            f'Summarize risk for {ctx["name"]}, {ctx["age"]}y, {ctx["diagnosis"]}, risk {ctx["risk_score"]:.0%}.'
        ),
        'B: Structured': (
            'You are a clinical AI for an ICU. Be specific and actionable in 3 sentences.',
            (f'Patient: {ctx["name"]}, {ctx["age"]}y, {ctx["diagnosis"]}. '
             f'Risk: {ctx["risk_score"]:.0%}. Drivers: {", ".join(ctx["drivers"])}. '
             f'HR {ctx["hr"]:.0f}, SpO2 {ctx["spo2"]:.1f}%, SBP {ctx["sbp"]:.0f}.')
        ),
        'C: Clinical Voice': (
            'You are an attending physician writing a handoff note. Use clinical shorthand. Recommend specific actions with timeframes.',
            (f'Quick summary for {ctx["name"]} ({ctx["age"]}y, {ctx["diagnosis"]}): '
             f'risk {ctx["risk_score"]:.0%}, NEWS2={ctx["news2"]}. '
             f'Primary concerns: {", ".join(ctx["drivers"])}. '
             f'HR {ctx["hr"]:.0f}, SpO2 {ctx["spo2"]:.1f}%, SBP {ctx["sbp"]:.0f}.')
        ),
        'D: Full Context': build_instruct_prompt(ctx),
    }
else:
    variants = {
        'A: Minimal':       (None, f'Patient {ctx["name"]}, risk {ctx["risk_score"]:.0%}. ICU assessment:'),
        'B: Structured':    (None, (f'ICU summary. {ctx["name"]}, {ctx["age"]}y, {ctx["diagnosis"]}, '
                                    f'{ctx["risk_score"]:.0%} risk. HR {ctx["hr"]:.0f}, SpO2 {ctx["spo2"]:.1f}%. Clinical concern:')),
        'C: Clinical Voice':(None, (f'Attending handoff — {ctx["name"]} ({ctx["diagnosis"]}): '
                                    f'Risk {ctx["risk_score"]:.0%}, NEWS2={ctx["news2"]}. '
                                    f'Primary: {ctx["drivers"][0]}. Recommendation:')),
        'D: Full Context':  (None, build_base_prompt(ctx)),
    }

def score_response(text, ctx):
    text_l = text.lower()
    words  = len(text.split())
    specific = sum(1 for t in [
        str(int(ctx['hr'])), f'{ctx["spo2"]:.0f}', str(int(ctx['sbp'])),
        ctx['diagnosis'].split()[0].lower(), ctx['risk_level'].lower(),
    ] if t in text_l)
    actions = sum(1 for w in ['recommend','consult','initiate','monitor','review',
                               'consider','urgent','immediate','within','hours'] if w in text_l)
    brevity = 3 if 40 <= words <= 85 else (2 if 25 <= words <= 130 else 1)
    return {'specific': specific, 'actions': min(actions,3), 'brevity': brevity,
            'words': words, 'total': specific + min(actions,3) + brevity}

print(f'{"Variant":<22} {"Score":>6} {"Words":>6} {"Specific":>9} {"Actions":>8} {"Brevity":>8}')
print('─' * 65)

all_results = {}
for name, (sys_p, usr_p) in variants.items():
    try:
        t0 = time.time()
        if MODEL_TYPE == 'instruct':
            out = call_instruct(sys_p, usr_p, max_tokens=200, temperature=0.3)
        else:
            out = call_base(usr_p, max_tokens=180, temperature=0.2)
        elapsed = time.time() - t0
        s = score_response(out, ctx)
        all_results[name] = {'text': out, 'score': s, 'time': elapsed}
        print(f'  {name:<20} {s["total"]:>6} {s["words"]:>6} {s["specific"]:>9} {s["actions"]:>8} {s["brevity"]:>8}  ({elapsed:.1f}s)')
    except Exception as e:
        print(f'  {name:<20} ERROR: {e}')

if all_results:
    best = max(all_results, key=lambda k: all_results[k]['score']['total'])
    print('─' * 65)
    print(f'  🏆 Best prompt: {best}')
    print()
    for line in textwrap.wrap(all_results[best]['text'], 63):
        print(f'    {line}')

In [ ]:
cmp_row = df[df['risk_level'] == 'HIGH'].nlargest(3, 'risk_score').iloc[1]
cmp_ctx = build_patient_context(cmp_row)

print(f'Comparison: {cmp_ctx["patient_id"]} — {cmp_ctx["name"]} — {cmp_ctx["risk_score"]:.0%}')
print()

def run_base(ctx):
    try:
        c = OpenAI(base_url=f'http://localhost:{BASE_PORT}/v1', api_key='none')
        r = c.completions.create(
            model='local-model', prompt=build_base_prompt(ctx),
            max_tokens=200, temperature=0.2, stop=['\n\nICU','\n\n=','Patient:']
        )
        return 'This patient presents with ' + r.choices[0].text.strip()
    except Exception as e:
        return f'[Base unavailable: {e}]'

def run_instruct(ctx):
    try:
        c = OpenAI(base_url=f'http://localhost:{INSTRUCT_PORT}/v1', api_key='none')
        sys_p, usr_p = build_instruct_prompt(ctx)
        r = c.chat.completions.create(
            model='local-model',
            messages=[{'role':'system','content':sys_p},{'role':'user','content':usr_p}],
            max_tokens=220, temperature=0.3
        )
        return r.choices[0].message.content.strip()
    except Exception as e:
        return f'[Instruct unavailable: {e}]'

print('Calling Base model...',     end=' ', flush=True)
t0=time.time(); base_out=run_base(cmp_ctx);    base_t=time.time()-t0; print(f'{base_t:.1f}s')

print('Calling Instruct model...', end=' ', flush=True)
t0=time.time(); inst_out=run_instruct(cmp_ctx); inst_t=time.time()-t0; print(f'{inst_t:.1f}s')

print()
print('=' * 65)
print(f'  PATIENT: {cmp_ctx["name"]}  |  Risk: {cmp_ctx["risk_score"]:.0%}  |  {cmp_ctx["diagnosis"]}')
print('=' * 65)
print()
print(f'   BASE MODEL ({base_t:.1f}s)')
print('  ' + '─' * 61)
for line in textwrap.wrap(base_out, 61): print(f'    {line}')
print()
print(f'   INSTRUCT MODEL ({inst_t:.1f}s)')
print('  ' + '─' * 61)
for line in textwrap.wrap(inst_out, 61): print(f'    {line}')
print()

if 'unavailable' not in base_out.lower() and 'unavailable' not in inst_out.lower():
    bs  = score_response(base_out, cmp_ctx)
    isc = score_response(inst_out, cmp_ctx)
    print(f'  {"Metric":<18} {"Base":>8} {"Instruct":>10}  Winner')
    print('  ' + '─' * 44)
    for m in ['specific','actions','brevity','total']:
        w = 'Base' if bs[m] > isc[m] else ('Instruct' if isc[m] > bs[m] else 'Tie')
        print(f'  {m.capitalize():<18} {bs[m]:>8} {isc[m]:>10}  {w}')
    rec = 'Instruct' if isc['total'] >= bs['total'] else 'Base'
    print(f'\n  Recommendation: {rec} model for clinical summaries')
print('=' * 65)

In [ ]:
if 'insights_df' not in dir() or len(insights_df) == 0:
    print('  No summaries to save — run Section 5 first.')
else:
    out_path = DATA_DIR / 'nemotron_insights.csv'
    insights_df.to_csv(out_path, index=False)
    print(f' Saved {len(insights_df)} summaries → {out_path}')

    stats = {
        'model_type':            MODEL_TYPE,
        'total_summaries':       len(insights_df),
        'high_risk_count':       int((insights_df['risk_level']=='HIGH').sum()),
        'moderate_risk_count':   int((insights_df['risk_level']=='MODERATE').sum()),
        'avg_generation_time_s': float(insights_df['generation_time_s'].mean()),
        'generated_at':          pd.Timestamp.now().isoformat(),
    }
    with open(DATA_DIR / 'nemotron_stats.json','w') as f:
        json.dump(stats, f, indent=2)
    print(f' Saved stats → {DATA_DIR / "nemotron_stats.json"}')
    print()
    print('Sample summaries:')
    print('─' * 62)
    for _, row in insights_df.head(3).iterrows():
        icon = '🔴' if row['risk_level']=='HIGH' else '🟡'
        print(f'{icon} {row["patient_id"]}  {row["name"]}  ({row["risk_score"]:.0%})')
        for line in textwrap.wrap(str(row['nemotron_insight']), 60)[:3]:
            print(f'   {line}')
        print()
    print('─' * 62)
    print()
    print('Dashboard loads nemotron_insights.csv automatically.')
    print('Launch: python dashboard/app.py  →  http://localhost:8050')

## 9 · Quick Reference

| Task | Command |
|------|---------|
| Start Base server | `llama-server --model ~/nemotron-nano/*.gguf --port 8001 --ctx-size 8192` |
| Start Instruct server | `llama-server --model ~/nemotron-nano-instruct/*.gguf --port 8002 --ctx-size 8192 --chat-template chatml` |
| Check server health | Run Section 1 |
| Regenerate all summaries | Re-run Section 5 |
| Switch Base ↔ Instruct | Change `MODEL_TYPE` in Section 0 |
| Use NVIDIA cloud API | Set `MOCK_MODE=False, LOCAL_MODE=False` in notebook 01 |

### Prompt Engineering Rules

**Base model:** Write the first words of the answer so the model continues in the right clinical voice. Always use `stop=["\n\n", "Patient:"]` to prevent it generating extra patients. Temperature `0.2` for demo consistency.

**Instruct model:** Put the clinical persona in `system`, patient data in `user`. Ask for a specific sentence count — `'in exactly 3 sentences'` works reliably. Temperature `0.3`.

**Demo day tip:** Pre-generate and export summaries via Section 8 the night before. The dashboard loads from CSV — zero model latency, no risk of the server being slow in the meeting room.